# Task 157: plasmapy_langmuir — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Langmuir probe analysis for plasma density/temperature

**Paper**: PlasmaPy: An open source Python package for plasma research
**Repository**: https://github.com/PlasmaPy/PlasmaPy

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: N/A (parameter estimation)
- **SSIM**: N/A
- **Evaluation**: Langmuir probe parameter extraction (T_e RE=0.37%, n_e RE=0.18%, V_p RE=0.27%)

### Mathematical Formulation

The general inverse problem seeks to recover $\mathbf{x}$ from indirect measurements:

$$\mathbf{y} = \mathcal{A}(\mathbf{x}) + \boldsymbol{\eta}$$

**Regularized solution**:
$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \frac{1}{2}\|\mathcal{A}(\mathbf{x}) - \mathbf{y}\|_2^2 + \lambda \mathcal{R}(\mathbf{x})$$

The regularization parameter $\lambda$ balances data fidelity against prior constraints, controlling the bias-variance trade-off.


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 157: plasmapy_langmuir — Langmuir Probe I-V Curve Inversion

Inverse Problem: Extract plasma parameters (T_e, n_e, V_p, V_f) from a
Langmuir probe I-V characteristic curve.

Physics:
  The current collected by a Langmuir probe immersed in a plasma depends on
  the applied bias voltage V:
    - V << V_p  (ion saturation):  I ≈ I_ion_sat  (negative, constant)
    - V < V_p   (transition):      I = I_ion_sat + I_e_sat * exp(e(V-V_p)/(k_B T_e))
    - V >= V_p  (electron sat.):   I = I_ion_sat + I_e_sat

  where I_e_sat = n_e * e * A_probe * sqrt(k_B T_e / (2π m_e))

Forward model parameters:
  T_e      — electron temperature  [eV]
  n_e      — electron density      [m⁻³]
  V_p      — plasma potential      [V]
  I_ion_sat — ion saturation current [A]

Inversion: nonlinear least-squares curve_fit on the I-V data.
"""

import os
import json
import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`electron_saturation_current`, `langmuir_iv`, `floating_potential`, `main`

In [ ]:
def electron_saturation_current(T_e_eV, n_e):
    """Electron saturation current [A] for given T_e (eV) and n_e (m⁻³)."""
    T_e_K = T_e_eV * EV_TO_K
    return n_e * E_CHARGE * A_PROBE * np.sqrt(K_BOLTZMANN * T_e_K / (2 * np.pi * M_ELECTRON))

def langmuir_iv(V, T_e, n_e, V_p, I_ion_sat):
    """
    Theoretical Langmuir probe I-V characteristic.

    Parameters
    ----------
    V : array_like   — bias voltage [V]
    T_e : float      — electron temperature [eV]
    n_e : float      — electron density [m⁻³]
    V_p : float      — plasma potential [V]
    I_ion_sat : float — ion saturation current [A] (negative)

    Returns
    -------
    I : ndarray — probe current [A]
    """
    T_e_K = T_e * EV_TO_K
    I_e_sat = electron_saturation_current(T_e, n_e)

    # Clamp the exponent to avoid overflow
    exponent = E_CHARGE * (V - V_p) / (K_BOLTZMANN * T_e_K)
    exponent = np.clip(exponent, -500, 500)

    I = np.where(
        V < V_p,
        I_ion_sat + I_e_sat * np.exp(exponent),
        I_ion_sat + I_e_sat,
    )
    return I

def floating_potential(T_e_eV, n_e, V_p, I_ion_sat):
    """Compute floating potential V_f where I(V_f) = 0."""
    I_e_sat = electron_saturation_current(T_e_eV, n_e)
    if I_e_sat <= 0 or -I_ion_sat <= 0:
        return V_p  # degenerate
    T_e_K = T_e_eV * EV_TO_K
    V_f = V_p + (K_BOLTZMANN * T_e_K / E_CHARGE) * np.log(-I_ion_sat / I_e_sat)
    return V_f

def main():
    out_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 60)
    print("Task 157: Langmuir Probe I-V Curve Inversion")
    print("=" * 60)

    results = run_all_cases()

    # ── Metrics ──
    metrics = {"test_cases": []}
    all_re = {"T_e": [], "n_e": [], "V_p": [], "I_ion_sat": [], "V_f": []}

    for r in results:
        case_metric = {
            "case": r["case"],
            "true": {
                "T_e": r["true_params"]["T_e"],
                "n_e": r["true_params"]["n_e"],
                "V_p": r["true_params"]["V_p"],
                "I_ion_sat": r["true_params"]["I_ion_sat"],
                "V_f": r["V_f_true"],
            },
            "estimated": {
                "T_e": r["fitted_params"]["T_e"],
                "n_e": r["fitted_params"]["n_e"],
                "V_p": r["fitted_params"]["V_p"],
                "I_ion_sat": r["fitted_params"]["I_ion_sat"],
                "V_f": r["fitted_params"]["V_f"],
            },
            "relative_errors": {k: f"{v*100:.4f}%" for k, v in r["relative_errors"].items()},
        }
        metrics["test_cases"].append(case_metric)

        for k in all_re:
            all_re[k].append(r["relative_errors"][k])

        print(f"\n--- Case: {r['case']} ---")
        print(f"  T_e : true={r['true_params']['T_e']:.2f} eV,  est={r['fitted_params']['T_e']:.4f} eV,  RE={r['relative_errors']['T_e']*100:.4f}%")
        print(f"  n_e : true={r['true_params']['n_e']:.2e} m⁻³, est={r['fitted_params']['n_e']:.4e} m⁻³, RE={r['relative_errors']['n_e']*100:.4f}%")
        print(f"  V_p : true={r['true_params']['V_p']:.2f} V,   est={r['fitted_params']['V_p']:.4f} V,   RE={r['relative_errors']['V_p']*100:.4f}%")
        print(f"  V_f : true={r['V_f_true']:.4f} V,   est={r['fitted_params']['V_f']:.4f} V,   RE={r['relative_errors']['V_f']*100:.4f}%")

    mean_re = {k: np.mean(v) * 100 for k, v in all_re.items()}
    metrics["mean_relative_errors"] = {k: f"{v:.4f}%" for k, v in mean_re.items()}
    metrics["overall_mean_RE"] = f"{np.mean(list(mean_re.values())):.4f}%"

    print(f"\n{'='*60}")
    print("Mean Relative Errors across all cases:")
    for k, v in mean_re.items():
        status = "✓ PASS" if v < 10 else "✗ FAIL"
        print(f"  {k:12s}: {v:.4f}%  {status}")
    overall = np.mean(list(mean_re.values()))
    print(f"  {'Overall':12s}: {overall:.4f}%  {'✓ PASS' if overall < 10 else '✗ FAIL'}")

    # ── Save outputs ──
    metrics_path = os.path.join(out_dir, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2, default=str)
    print(f"\n[INFO] Saved metrics → {metrics_path}")

    # Ground truth: dict of all true parameters for baseline case
    gt = {
        "T_e": results[0]["true_params"]["T_e"],
        "n_e": results[0]["true_params"]["n_e"],
        "V_p": results[0]["true_params"]["V_p"],
        "I_ion_sat": results[0]["true_params"]["I_ion_sat"],
        "V_f": results[0]["V_f_true"],
        "I_V_data": {
            "V": results[0]["V"].tolist(),
            "I_clean": results[0]["I_clean"].tolist(),
            "I_noisy": results[0]["I_noisy"].tolist(),
        },
    }
    gt_path = os.path.join(out_dir, "ground_truth.npy")
    np.save(gt_path, gt, allow_pickle=True)
    print(f"[INFO] Saved ground truth → {gt_path}")

    # Reconstruction: dict of fitted parameters for baseline case
    recon = {
        "T_e": results[0]["fitted_params"]["T_e"],
        "n_e": results[0]["fitted_params"]["n_e"],
        "V_p": results[0]["fitted_params"]["V_p"],
        "I_ion_sat": results[0]["fitted_params"]["I_ion_sat"],
        "V_f": results[0]["fitted_params"]["V_f"],
        "I_V_data": {
            "V": results[0]["V"].tolist(),
            "I_fitted": results[0]["I_fitted"].tolist(),
        },
    }
    recon_path = os.path.join(out_dir, "reconstruction.npy")
    np.save(recon_path, recon, allow_pickle=True)
    print(f"[INFO] Saved reconstruction → {recon_path}")

    # Visualization
    plot_path = os.path.join(out_dir, "reconstruction_result.png")
    plot_results(results, plot_path)

    print(f"\n{'='*60}")
    print("Task 157 COMPLETE")
    print(f"{'='*60}")

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `synthesize_iv`

In [ ]:
def synthesize_iv(T_e, n_e, V_p, I_ion_sat, V_range=(-30, 30), n_points=500,
                  noise_level=0.02, seed=42):
    """Generate noisy Langmuir probe I-V data from known parameters."""
    rng = np.random.default_rng(seed)
    V = np.linspace(V_range[0], V_range[1], n_points)
    I_clean = langmuir_iv(V, T_e, n_e, V_p, I_ion_sat)
    noise_amplitude = noise_level * (np.max(I_clean) - np.min(I_clean))
    noise = rng.normal(0, noise_amplitude, size=I_clean.shape)
    I_noisy = I_clean + noise
    return V, I_clean, I_noisy

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `invert_iv`, `run_all_cases`

In [ ]:
def invert_iv(V, I_noisy, p0=None):
    """
    Fit the Langmuir I-V model to noisy data.

    Returns
    -------
    params : dict with keys T_e, n_e, V_p, I_ion_sat, V_f
    """
    if p0 is None:
        # Heuristic initial guesses
        I_min = np.min(I_noisy)
        I_max = np.max(I_noisy)
        V_p_guess = V[np.argmax(np.gradient(I_noisy, V))]
        T_e_guess = 5.0
        n_e_guess = 1e17
        I_ion_sat_guess = I_min
        p0 = [T_e_guess, n_e_guess, V_p_guess, I_ion_sat_guess]

    bounds = (
        [0.1, 1e14, V.min(), -1.0],    # lower
        [100.0, 1e20, V.max(), 0.0],     # upper
    )

    popt, pcov = curve_fit(
        langmuir_iv, V, I_noisy,
        p0=p0,
        bounds=bounds,
        maxfev=50000,
        method="trf",
    )

    T_e_fit, n_e_fit, V_p_fit, I_ion_sat_fit = popt
    V_f_fit = floating_potential(T_e_fit, n_e_fit, V_p_fit, I_ion_sat_fit)
    perr = np.sqrt(np.diag(pcov))

    return {
        "T_e": T_e_fit,
        "n_e": n_e_fit,
        "V_p": V_p_fit,
        "I_ion_sat": I_ion_sat_fit,
        "V_f": V_f_fit,
        "std_errors": {
            "T_e": perr[0],
            "n_e": perr[1],
            "V_p": perr[2],
            "I_ion_sat": perr[3],
        },
    }

def run_all_cases():
    """Run forward + inverse on every test case, return aggregated results."""
    all_results = []
    for i, tc in enumerate(TEST_CASES):
        V, I_clean, I_noisy = synthesize_iv(
            tc["T_e"], tc["n_e"], tc["V_p"], tc["I_ion_sat"],
            noise_level=0.02, seed=42 + i,
        )
        fitted = invert_iv(V, I_noisy)

        V_f_true = floating_potential(tc["T_e"], tc["n_e"], tc["V_p"], tc["I_ion_sat"])
        I_fitted = langmuir_iv(V, fitted["T_e"], fitted["n_e"], fitted["V_p"], fitted["I_ion_sat"])

        re = {
            "T_e": relative_error(tc["T_e"], fitted["T_e"]),
            "n_e": relative_error(tc["n_e"], fitted["n_e"]),
            "V_p": relative_error(tc["V_p"], fitted["V_p"]),
            "I_ion_sat": relative_error(tc["I_ion_sat"], fitted["I_ion_sat"]),
            "V_f": relative_error(V_f_true, fitted["V_f"]),
        }

        all_results.append({
            "case": tc["label"],
            "true_params": tc,
            "fitted_params": fitted,
            "V_f_true": V_f_true,
            "relative_errors": re,
            "V": V,
            "I_clean": I_clean,
            "I_noisy": I_noisy,
            "I_fitted": I_fitted,
        })
    return all_results

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `relative_error`, `plot_results`

In [ ]:
def relative_error(true_val, est_val):
    return abs(est_val - true_val) / abs(true_val) if true_val != 0 else 0.0

def plot_results(results, save_path):
    """4-panel visualization for the baseline case + summary bar chart."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Task 157: Langmuir Probe I-V Curve Inversion", fontsize=14, fontweight="bold")

    # Use baseline case (index 0) for panels 1-3
    r = results[0]
    V = r["V"]
    I_noisy = r["I_noisy"]
    I_clean = r["I_clean"]
    I_fitted = r["I_fitted"]

    # Panel 1: Noisy I-V data
    ax = axes[0, 0]
    ax.scatter(V, I_noisy * 1e3, s=2, alpha=0.5, color="steelblue", label="Noisy data")
    ax.set_xlabel("Bias Voltage V [V]")
    ax.set_ylabel("Current I [mA]")
    ax.set_title("(a) Noisy I-V Data")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Panel 2: True vs Fitted curves
    ax = axes[0, 1]
    ax.plot(V, I_clean * 1e3, "k-", lw=2, label="Ground truth")
    ax.plot(V, I_fitted * 1e3, "r--", lw=2, label="Fitted")
    ax.scatter(V, I_noisy * 1e3, s=1, alpha=0.3, color="gray", label="Noisy data")
    ax.set_xlabel("Bias Voltage V [V]")
    ax.set_ylabel("Current I [mA]")
    ax.set_title("(b) True vs Fitted I-V Curve")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Panel 3: Residuals
    ax = axes[1, 0]
    residuals = (I_fitted - I_clean) * 1e3
    ax.plot(V, residuals, "g-", lw=1)
    ax.axhline(0, color="k", ls="--", lw=0.5)
    ax.set_xlabel("Bias Voltage V [V]")
    ax.set_ylabel("Residual [mA]")
    ax.set_title("(c) Fit Residuals (Fitted − True)")
    ax.grid(True, alpha=0.3)

    # Panel 4: Parameter comparison bar chart (all cases)
    ax = axes[1, 1]
    param_names = ["T_e", "n_e", "V_p", "I_ion_sat"]
    mean_re = {}
    for pn in param_names:
        vals = [r2["relative_errors"][pn] * 100 for r2 in results]
        mean_re[pn] = np.mean(vals)
    x = np.arange(len(param_names))
    colors = ["#2196F3" if v < 5 else "#FF9800" if v < 10 else "#F44336" for v in mean_re.values()]
    bars = ax.bar(x, list(mean_re.values()), color=colors, edgecolor="k", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(["$T_e$", "$n_e$", "$V_p$", "$I_{ion,sat}$"])
    ax.set_ylabel("Mean Relative Error [%]")
    ax.set_title(f"(d) Mean RE Across {len(results)} Cases")
    ax.axhline(10, color="red", ls="--", lw=1, label="10% threshold")
    for bar, val in zip(bars, mean_re.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f"{val:.2f}%", ha="center", va="bottom", fontsize=9)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[INFO] Saved plot → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
out_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
os.makedirs(out_dir, exist_ok=True)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print("=" * 60)
print("Task 157: Langmuir Probe I-V Curve Inversion")
print("=" * 60)

results = run_all_cases()

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── Metrics ──
metrics = {"test_cases": []}
all_re = {"T_e": [], "n_e": [], "V_p": [], "I_ion_sat": [], "V_f": []}

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
for r in results:
    case_metric = {
        "case": r["case"],
        "true": {
            "T_e": r["true_params"]["T_e"],
            "n_e": r["true_params"]["n_e"],
            "V_p": r["true_params"]["V_p"],
            "I_ion_sat": r["true_params"]["I_ion_sat"],
            "V_f": r["V_f_true"],
        },
        "estimated": {
            "T_e": r["fitted_params"]["T_e"],
            "n_e": r["fitted_params"]["n_e"],
            "V_p": r["fitted_params"]["V_p"],
            "I_ion_sat": r["fitted_params"]["I_ion_sat"],
            "V_f": r["fitted_params"]["V_f"],
        },
        "relative_errors": {k: f"{v*100:.4f}%" for k, v in r["relative_errors"].items()},
    }
    metrics["test_cases"].append(case_metric)

for k in all_re:
        all_re[k].append(r["relative_errors"][k])

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print(f"\n--- Case: {r['case']} ---")
    print(f"  T_e : true={r['true_params']['T_e']:.2f} eV,  est={r['fitted_params']['T_e']:.4f} eV,  RE={r['relative_errors']['T_e']*100:.4f}%")
    print(f"  n_e : true={r['true_params']['n_e']:.2e} m⁻³, est={r['fitted_params']['n_e']:.4e} m⁻³, RE={r['relative_errors']['n_e']*100:.4f}%")
    print(f"  V_p : true={r['true_params']['V_p']:.2f} V,   est={r['fitted_params']['V_p']:.4f} V,   RE={r['relative_errors']['V_p']*100:.4f}%")
    print(f"  V_f : true={r['V_f_true']:.4f} V,   est={r['fitted_params']['V_f']:.4f} V,   RE={r['relative_errors']['V_f']*100:.4f}%")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
mean_re = {k: np.mean(v) * 100 for k, v in all_re.items()}
metrics["mean_relative_errors"] = {k: f"{v:.4f}%" for k, v in mean_re.items()}
metrics["overall_mean_RE"] = f"{np.mean(list(mean_re.values())):.4f}%"

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print(f"\n{'='*60}")
print("Mean Relative Errors across all cases:")
for k, v in mean_re.items():
    status = "✓ PASS" if v < 10 else "✗ FAIL"
    print(f"  {k:12s}: {v:.4f}%  {status}")
overall = np.mean(list(mean_re.values()))
print(f"  {'Overall':12s}: {overall:.4f}%  {'✓ PASS' if overall < 10 else '✗ FAIL'}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── Save outputs ──
metrics_path = os.path.join(out_dir, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\n[INFO] Saved metrics → {metrics_path}")

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# Ground truth: dict of all true parameters for baseline case
gt = {
    "T_e": results[0]["true_params"]["T_e"],
    "n_e": results[0]["true_params"]["n_e"],
    "V_p": results[0]["true_params"]["V_p"],
    "I_ion_sat": results[0]["true_params"]["I_ion_sat"],
    "V_f": results[0]["V_f_true"],
    "I_V_data": {
        "V": results[0]["V"].tolist(),
        "I_clean": results[0]["I_clean"].tolist(),
        "I_noisy": results[0]["I_noisy"].tolist(),
    },
}
gt_path = os.path.join(out_dir, "ground_truth.npy")
np.save(gt_path, gt, allow_pickle=True)
print(f"[INFO] Saved ground truth → {gt_path}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Reconstruction: dict of fitted parameters for baseline case
recon = {
    "T_e": results[0]["fitted_params"]["T_e"],
    "n_e": results[0]["fitted_params"]["n_e"],
    "V_p": results[0]["fitted_params"]["V_p"],
    "I_ion_sat": results[0]["fitted_params"]["I_ion_sat"],
    "V_f": results[0]["fitted_params"]["V_f"],
    "I_V_data": {
        "V": results[0]["V"].tolist(),
        "I_fitted": results[0]["I_fitted"].tolist(),
    },
}
recon_path = os.path.join(out_dir, "reconstruction.npy")
np.save(recon_path, recon, allow_pickle=True)
print(f"[INFO] Saved reconstruction → {recon_path}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Visualization
plot_path = os.path.join(out_dir, "reconstruction_result.png")
plot_results(results, plot_path)

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print(f"\n{'='*60}")
print("Task 157 COMPLETE")
print(f"{'='*60}")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **plasmapy_langmuir**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=N/A (parameter estimation), SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: PlasmaPy: An open source Python package for plasma research
- Repository: https://github.com/PlasmaPy/PlasmaPy
